In [1]:
import pandas as pd
import numpy as np
import scanpy as sc
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from scipy.spatial.distance import cdist
import torch.nn.functional as F


In [4]:
adata = sc.read_h5ad('../example.h5ad')

In [5]:
adata.obs['X'] = adata.obs['array_row'] *100
adata.obs['Y'] = adata.obs['array_col'] *100
adata.obs

,in_tissue,array_row,array_col,n_counts,X,Y
AAACAAGTATCTCCCA-1,1,50,102,5625.0,5000,10200
AAACACCAATAACTGC-1,1,59,19,9159.0,5900,1900
AAACAGAGCGACTCCT-1,1,14,94,2660.0,1400,9400
AAACAGGGTCTATATT-1,1,47,13,9020.0,4700,1300
AAACAGTGTTCCTGGG-1,1,73,43,9714.0,7300,4300
...,...,...,...,...,...,...
TTGTTGTGTGTCAAGA-1,1,31,77,7260.0,3100,7700
TTGTTTCACATCCAGG-1,1,58,42,7510.0,5800,4200
TTGTTTCATTAGTCTA-1,1,60,30,5742.0,6000,3000
TTGTTTCCATACAACT-1,1,45,27,20118.0,4500,2700


In [6]:
adata.obs['tcr'] = np.random.randint(2, size=adata.n_obs)

In [7]:
adata.obs

,in_tissue,array_row,array_col,n_counts,X,Y,tcr
AAACAAGTATCTCCCA-1,1,50,102,5625.0,5000,10200,1
AAACACCAATAACTGC-1,1,59,19,9159.0,5900,1900,0
AAACAGAGCGACTCCT-1,1,14,94,2660.0,1400,9400,1
AAACAGGGTCTATATT-1,1,47,13,9020.0,4700,1300,1
AAACAGTGTTCCTGGG-1,1,73,43,9714.0,7300,4300,0
...,...,...,...,...,...,...,...
TTGTTGTGTGTCAAGA-1,1,31,77,7260.0,3100,7700,1
TTGTTTCACATCCAGG-1,1,58,42,7510.0,5800,4200,1
TTGTTTCATTAGTCTA-1,1,60,30,5742.0,6000,3000,1
TTGTTTCCATACAACT-1,1,45,27,20118.0,4500,2700,0


In [8]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cpu


In [9]:
class BagsDataset(Dataset):
    def __init__(self, adata, radius=50):
        self.bags = self.create_bags(adata, radius)

    def __len__(self):
        return len(self.bags)

    def __getitem__(self, idx):
        bag = self.bags[idx]
        instances = bag['instances']
        label = bag['label']
        return instances, label

    def create_bags(self, adata, radius):
        spatial_coords_x = adata.obs['X']
        spatial_coords_y = adata.obs['Y']
        spatial_coords = np.array(list(zip(spatial_coords_x, spatial_coords_y)))

        gene_expression = adata.X
        labels = adata.obs['tcr'].values
        bags = {}

        dist_matrix = cdist(spatial_coords, spatial_coords, metric='euclidean')

        for i in range(len(spatial_coords)):
            in_circle = np.where(dist_matrix[i] <= radius)[0]

            bag_data = gene_expression[in_circle].todense()  # Convert sparse matrix to dense
            bag_label = labels[i]

            bags[i] = {
                'instances': bag_data,
                'label': bag_label
            }
            print(f"Bag {i} contains {len(in_circle)} instances")

        return bags

In [10]:
dataset = BagsDataset(adata, radius=200)
dataset.__getitem__(0)

Bag 0 contains 9 instances
Bag 1 contains 9 instances
Bag 2 contains 9 instances
Bag 3 contains 7 instances
Bag 4 contains 5 instances
Bag 5 contains 9 instances
Bag 6 contains 5 instances
Bag 7 contains 9 instances
Bag 8 contains 8 instances
Bag 9 contains 9 instances
Bag 10 contains 9 instances
Bag 11 contains 9 instances
Bag 12 contains 4 instances
Bag 13 contains 9 instances
Bag 14 contains 9 instances
Bag 15 contains 8 instances
Bag 16 contains 8 instances
Bag 17 contains 9 instances
Bag 18 contains 9 instances
Bag 19 contains 9 instances
Bag 20 contains 9 instances
Bag 21 contains 8 instances
Bag 22 contains 8 instances
Bag 23 contains 9 instances
Bag 24 contains 9 instances
Bag 25 contains 9 instances
Bag 26 contains 9 instances
Bag 27 contains 9 instances
Bag 28 contains 9 instances
Bag 29 contains 9 instances
Bag 30 contains 9 instances
Bag 31 contains 9 instances
Bag 32 contains 9 instances
Bag 33 contains 9 instances
Bag 34 contains 9 instances
Bag 35 contains 9 instances
Ba

(matrix([[0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         ...,
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.],
         [0., 0., 0., ..., 0., 0., 0.]], dtype=float32),
 np.int64(1))

In [11]:
def custom_collate_fn(batch):
    # Custom collate function to handle bags with variable number of instances
    instances, labels = zip(*batch)
    return instances, torch.tensor(labels[0], dtype=torch.float32).view(1)

In [12]:
dataloader = DataLoader(dataset, batch_size=16, shuffle=True, collate_fn=custom_collate_fn)
len(dataloader)

165

In [13]:
class Attention(nn.Module):
    def __init__(self):
        super(Attention, self).__init__()
        self.M = 500
        self.L = 128

        self.feature_extractor_part1 = nn.Sequential(
            nn.Linear(1, 128),  # 使用全连接层
            nn.ReLU(),
            nn.Linear(128, self.M),
            nn.ReLU()
        )

        self.attention = nn.Sequential(
            nn.Linear(self.M, self.L),  # matrix V
            nn.Tanh(),
            nn.Linear(self.L, 1)  # single attention weight
        )

        self.classifier = nn.Sequential(
            nn.Linear(self.M, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        H = self.feature_extractor_part1(x)
        A = self.attention(H)  # Kx1
        A = torch.transpose(A, 1, 0)  # 1xK
        A = F.softmax(A, dim=1)  # softmax over K

        Z = torch.mm(A, H)  # 1xM

        Y_prob = self.classifier(Z)
        Y_hat = torch.ge(Y_prob, 0.5).float()

        return Y_prob, Y_hat, A


In [14]:
class MIL(nn.Module):
    def __init__(self, input_dim):
        super(MIL, self).__init__()
        self.fc1 = nn.Linear(input_dim, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.sigmoid(self.fc3(x))
        return x


In [15]:
class MILAggregator(nn.Module):
    def __init__(self, input_dim):
        super(MILAggregator, self).__init__()
        self.mil = MIL(input_dim)
        self.attention = Attention()
        self.stored_weights = []

    def forward(self, bag):
        # Process each instance in the bag
        instance_outputs = torch.stack([self.mil(instance) for instance in bag])

        # Reshape instance_outputs to (num_instances, 1)
        instance_outputs = instance_outputs.view(-1, 1)

        # Debug print
        #print("instance_outputs shape:", instance_outputs.shape)

        # Calculate attention weights
        Y_prob, Y_hat, weights = self.attention(instance_outputs)
        
        # Debug print
        #print("weights shape:", weights.shape)
        #print("instance_outputs shape after attention:", instance_outputs.shape)
        
        # Aggregate the instance outputs using attention weights
        bag_output = torch.sum(weights * instance_outputs.T, dim=1, keepdim=True)

        self.stored_weights.append(weights.detach().cpu().numpy())
        
        # Debug print
        #print("bag_output shape before view:", bag_output.shape)

        return bag_output.view(1)  # Ensure the output shape is a scalar
    
    def get_stored_weights(self):
        return self.stored_weights
    
    def clear_stored_weights(self):
        self.stored_weights = []


In [16]:
model= MILAggregator(adata.n_vars)
model.to(device)


MILAggregator(
  (mil): MIL(
    (fc1): Linear(in_features=19379, out_features=64, bias=True)
    (fc2): Linear(in_features=64, out_features=32, bias=True)
    (fc3): Linear(in_features=32, out_features=1, bias=True)
  )
  (attention): Attention(
    (feature_extractor_part1): Sequential(
      (0): Linear(in_features=1, out_features=128, bias=True)
      (1): ReLU()
      (2): Linear(in_features=128, out_features=500, bias=True)
      (3): ReLU()
    )
    (attention): Sequential(
      (0): Linear(in_features=500, out_features=128, bias=True)
      (1): Tanh()
      (2): Linear(in_features=128, out_features=1, bias=True)
    )
    (classifier): Sequential(
      (0): Linear(in_features=500, out_features=1, bias=True)
      (1): Sigmoid()
    )
  )
)

In [17]:
criterion = nn.BCELoss().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)


In [18]:
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for i, (instances, label) in enumerate(dataloader):
        instances = [torch.tensor(np.array(instance), dtype=torch.float32).to(device) for instance in instances[0]]
        label = torch.tensor(label, dtype=torch.float32).to(device)

        optimizer.zero_grad()

       
        output = model(instances)


        loss = criterion(output, label)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        if i % 10 == 9:  
            print(f"Epoch [{epoch+1}/{num_epochs}], Step [{i+1}], Loss: {loss.item():.4f}")

    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss / len(dataloader):.4f}")

print("Finished Training")


/tmp/ipykernel_52482/2076624706.py:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  label = torch.tensor(label, dtype=torch.float32).to(device)


Epoch [1/10], Step [10], Loss: 0.0015
Epoch [1/10], Step [20], Loss: 1.0550
Epoch [1/10], Step [30], Loss: 1.2601
Epoch [1/10], Step [40], Loss: 0.1122
Epoch [1/10], Step [50], Loss: 0.0937
Epoch [1/10], Step [60], Loss: 0.4024
Epoch [1/10], Step [70], Loss: 1.7970
Epoch [1/10], Step [80], Loss: 0.0937
Epoch [1/10], Step [90], Loss: 0.3625
Epoch [1/10], Step [100], Loss: 1.5140
Epoch [1/10], Step [110], Loss: 0.1406
Epoch [1/10], Step [120], Loss: 0.6578
Epoch [1/10], Step [130], Loss: 0.8818
Epoch [1/10], Step [140], Loss: 0.5066
Epoch [1/10], Step [150], Loss: 0.8036
Epoch [1/10], Step [160], Loss: 0.3417
Epoch [1/10], Loss: 0.8681
Epoch [2/10], Step [10], Loss: 0.0625
Epoch [2/10], Step [20], Loss: 0.6838
Epoch [2/10], Step [30], Loss: 0.8119
Epoch [2/10], Step [40], Loss: 0.7201
Epoch [2/10], Step [50], Loss: 0.6458
Epoch [2/10], Step [60], Loss: 0.6347
Epoch [2/10], Step [70], Loss: 0.4626
Epoch [2/10], Step [80], Loss: 0.2303
Epoch [2/10], Step [90], Loss: 0.6848
Epoch [2/10], St

In [19]:
torch.save(model.state_dict(), 'mil_aggregator.pth')

In [20]:
def evaluate_model(model, dataloader, device):
    model.eval()
    total = 0
    correct = 0
    all_stored_weights = []

    with torch.no_grad():
        for idx, (instances, label) in enumerate(dataloader):
            instances = [torch.tensor(np.array(instance), dtype=torch.float32).to(device) for instance in instances[0]]
            label = torch.tensor(label, dtype=torch.float32).to(device)

            output = model(instances)
            predicted = (output > 0.5).float()
            total += label.size(0)
            correct += (predicted == label).sum().item()

            stored_weights = model.get_stored_weights()
            all_stored_weights.append(stored_weights)
            model.clear_stored_weights() 

    accuracy = 100 * correct / total
    print(f'Accuracy of the model on the entire dataset: {accuracy}%')

    for i, stored_weights in enumerate(all_stored_weights):
        for j, weights in enumerate(stored_weights):
            print(f"Data point {i}, Bag {j} weights: {weights}")
            



In [23]:
test_dataloader = DataLoader(dataset, batch_size=1, shuffle=True, collate_fn=custom_collate_fn)

In [24]:
evaluate_model(model, list(test_dataloader)[0:1], device)

Accuracy of the model on the entire dataset: 0.0%
Data point 0, Bag 0 weights: [[1.1829333e-06 1.4285684e-01 1.4285684e-01 1.4285684e-01 1.4285684e-01
  1.0520686e-06 1.4285684e-01 1.4285684e-01 1.4285684e-01]]


/tmp/ipykernel_52482/208287778.py:10: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  label = torch.tensor(label, dtype=torch.float32).to(device)
